In [2]:
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\dell\myenv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\dell\myenv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\dell\myenv\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\Users\dell\myenv\Lib\site-packages\tornado\platfor

In [3]:
import numpy as np

def randn_from_mt19937(shape, seed, mean=0.0, std=1.0, dtype=torch.float32, device='cpu'):
    """
    Generate a PyTorch tensor with normally distributed values using NumPy's MT19937 RNG.

    Args:
        shape (tuple): Shape of the tensor.
        seed (int): Seed for reproducibility.
        mean (float): Mean of the normal distribution.
        std (float): Standard deviation of the normal distribution.
        dtype (torch.dtype): Data type of the output tensor.
        device (str or torch.device): Device to place the tensor on.

    Returns:
        torch.Tensor: Tensor with normally distributed values.
    """
    size = np.prod(shape)
    rng = np.random.Generator(np.random.MT19937(seed))
    samples = rng.normal(loc=mean, scale=std, size=size).astype(np.float32)
    tensor = torch.from_numpy(samples).to(dtype=dtype).reshape(shape).to(device)
    return tensor


In [3]:
import torch
import torch.nn.functional as F

# --- 1. Define Input Tensor ---
# The input tensor for Conv1d should have the shape: (N, C_in, L_in)
# N: Batch size (number of samples)
# C_in: Number of input channels (e.g., embedding dimension)
# L_in: Length of the sequence

# Create a sample input tensor with random data
input_tensor =[
  [
    [
      [
        [-0.516964, 1.22192, 0.721333, 0.869636, 1.61822], 
        [1.58856, -1.18831, -0.187125, -0.687744, -0.0634201], 
        [0.849739, -0.385892, 0.297398, -1.05039, 1.32076], 
        [0.957779, -1.17469, 0.435903, -1.55086, 0.884557], 
        [0.0114265, -0.227518, -0.0544499, -0.381592, -1.84862]
      ], 
      [
        [0.219374, 1.63442, -1.46325, -0.641899, -1.28357], 
        [-0.0940366, -0.876347, 1.28242, 0.532373, -1.23327], 
        [-0.482664, 0.0108072, 0.366883, 2.40091, -0.863495], 
        [0.954369, 1.60008, -1.26013, -0.189323, 0.493696], 
        [0.0671027, 1.21292, -0.450653, -0.299507, -0.200332]
      ], 
      [
        [0.780412, -1.09702, -0.606103, 1.85599, -0.939128], 
        [0.27331, -0.01844, 1.27388, 0.787191, -0.209509], 
        [-0.574593, -0.756923, 0.760755, 1.19533, -1.76639], 
        [-0.702134, 0.100309, -1.45984, 1.52588, 2.21157], 
        [1.4996, -0.273863, 0.619165, -0.147712, 0.66358]
      ], 
      [
        [1.54048, 0.560455, -0.230439, -0.83715, 0.143307], 
        [-1.40409, 0.0962218, -0.459399, 0.906159, -0.64203], 
        [1.42637, -1.7264, 0.29969, 1.20096, 0.316924], 
        [-0.0962328, 0.312557, -1.52287, 0.744068, -1.02266], 
        [-0.0083384, 0.233371, -0.849648, -0.524936, -0.663464]
      ], 
      [
        [-0.566608, 1.01952, 0.122441, -0.335384, 0.824665], 
        [2.24651, -0.391692, 0.0788633, 0.843509, -0.899953], 
        [-0.31621, -1.06146, -1.66251, -0.215853, 1.08075], 
        [0.164415, -1.19123, 0.639323, -1.79431, 1.52739], 
        [-0.511735, -0.921379, -0.752286, -0.149504, 0.97621]
      ]
    ], 
    [
      [
        [0.435183, 1.04108, 0.359734, 0.383777, -0.0783321], 
        [-0.29086, 0.948444, 0.589491, 0.0714214, -1.59169], 
        [-2.08606, 0.406469, -0.862891, -0.128816, -0.0832325], 
        [0.0548126, -0.0418035, -0.641318, 0.947682, -1.20284], 
        [-0.848487, 1.72833, -2.20744, -0.0981202, 1.3058]
      ], 
      [
        [0.422576, -1.04532, 0.288402, -0.0623031, 0.123397], 
        [1.44192, -2.48644, -0.764208, 0.999384, 0.298426], 
        [0.866496, -1.18511, 0.206517, 0.676666, -0.149148], 
        [0.287263, -0.568569, 0.590687, -0.176682, 0.770473], 
        [0.207245, 0.0136186, 1.09357, -0.890917, -1.11435]
      ], 
      [
        [2.12645, 0.324284, 1.47549, -1.23752, -1.77466], 
        [1.66473, 0.478366, -0.125689, -0.560198, 1.88588], 
        [-0.703395, 0.730192, 0.983973, 1.21028, -0.243826], 
        [1.36622, 0.713993, -0.492485, -1.11412, -0.953354], 
        [-1.37982, 0.650501, -1.44723, -0.949764, 1.60879]
      ], 
      [
        [-0.669162, -1.6002, 0.020242, 0.852093, 0.670735], 
        [-0.931605, 0.398951, 0.113936, -0.204308, 1.40648], 
        [-2.47335, 1.11562, -0.517231, -0.591377, 0.602422], 
        [-0.136095, 1.03707, 0.0327517, 0.117034, -0.366917], 
        [-1.48255, -0.133674, -0.382009, 0.318783, -0.13]
      ], 
      [
        [-1.13096, 0.698333, -0.383805, -0.441243, 0.647358], 
        [0.592897, 1.07894, 1.00046, -0.312299, -0.223474], 
        [0.215738, 0.904944, -1.98682, -0.620351, -0.514702], 
        [-0.963428, -0.0390711, -1.11038, 0.591594, -0.231969], 
        [-1.46984, 1.07156, 0.572037, -0.74782, -1.41862]
      ]
    ]
  ], 
  [
    [
      [
        [2.90545, 2.12909, -0.631875, -0.131487, 1.54169], 
        [-0.783122, 0.431998, -0.301335, 0.626292, 0.844002], 
        [1.0889, 0.411229, 0.885492, 1.00876, 0.158174], 
        [-0.352039, 0.209676, -0.169676, -0.0514377, 0.545956], 
        [-0.16385, 2.45949, -0.474937, 0.720491, -1.00266]
      ], 
      [
        [0.277528, -1.16767, 1.14657, -1.5175, -0.933608], 
        [-0.500297, -1.56106, 0.563424, 1.95007, -1.64294], 
        [-0.600347, -0.112102, 0.837057, -2.69334, -1.89441], 
        [-1.05265, -0.420943, 0.57271, -1.2647, 0.837908], 
        [-0.430761, 0.212775, 0.603403, -0.127745, 1.35226]
      ], 
      [
        [0.430532, 1.16225, -0.802982, -0.957965, 0.389003], 
        [0.0567948, 1.44404, 1.10938, -0.0134231, -0.111697], 
        [-1.38074, 0.912554, 0.512877, 2.91479, -0.555727], 
        [-0.642663, -0.0349858, -0.716337, -0.852762, -0.807225], 
        [-0.0411759, -0.17756, 0.327586, 0.180025, -0.473449]
      ], 
      [
        [-0.579264, 0.683415, 0.953973, -0.910408, -0.226089], 
        [0.0631665, 0.29888, -0.487007, -1.41302, 1.21214], 
        [0.217646, -0.245418, -1.54972, 0.168696, 1.07765], 
        [-1.41041, -0.652929, -0.146751, 0.115298, 1.65441], 
        [0.501884, -1.65767, -0.825307, 0.203816, -0.227729]
      ], 
      [
        [-0.0380787, -0.37804, 0.680684, 0.815102, 0.672241], 
        [-0.440431, -0.48342, 0.195515, -1.21527, 0.395618], 
        [0.397159, 0.2058, 1.53273, -1.10358, -0.686421], 
        [-0.806079, -1.64404, -0.228938, 0.237062, 0.556594], 
        [-2.2522, 1.03752, -0.436526, 1.7181, -0.450076]
      ]
    ], 
    [
      [
        [0.514503, 0.824048, -0.70246, 1.04592, 1.42899], 
        [-0.167399, 0.278383, 0.649517, -0.742612, 0.267558], 
        [0.1339, -1.16002, -2.0782, -0.254266, -1.52884], 
        [0.0336459, -1.34662, -0.595901, -1.69836, -0.242858], 
        [0.575486, 1.09261, 0.575721, -0.346358, -1.11722]
      ], 
      [
        [2.31797, 0.248481, -0.515774, 0.0622256, -0.788661], 
        [-1.02772, -0.511089, -1.20156, -1.6797, -0.302122], 
        [-0.316767, 1.06388, -0.289162, -0.605724, -0.527166], 
        [-0.607109, -0.233354, -2.78008, -0.686034, -0.0639906], 
        [-0.918892, -1.01371, -1.33172, 0.615177, 0.406909]
      ], 
      [
        [-0.104334, 1.99209, -1.14657, 1.08259, 0.205844], 
        [-0.822957, -0.173725, 2.00064, 0.411792, 0.407158], 
        [0.239234, 0.961821, 0.818067, -0.740474, 0.0331249], 
        [-0.333931, -0.523291, -0.507972, 0.457922, -1.35327], 
        [-0.266868, -0.960509, -0.10699, 0.0372803, 0.771482]
      ], 
      [
        [0.630006, 0.627854, -0.753933, 0.305039, -0.191716], 
        [-1.02154, -0.135375, -0.195122, 0.271089, 0.558307], 
        [0.0943643, -1.67145, -0.449214, -1.52409, -0.036661], 
        [0.0103457, -0.0547148, 1.91864, -1.66052, -1.10318], 
        [0.557238, 0.787936, -1.46632, -1.263, 1.64635]
      ], 
      [
        [-1.18114, -0.748972, -1.69712, 0.756976, -0.989778], 
        [-1.31725, 0.69877, -1.68927, 0.096982, -0.324205], 
        [-0.0683886, -0.747694, -0.271226, -0.780501, 1.02708], 
        [0.149129, -0.474722, -0.884713, 1.16156, -0.657655], 
        [0.564306, 0.459052, -0.233192, -1.07673, 0.925459]
      ]
    ]
  ], 
  [
    [
      [
        [-1.77747, 0.0738024, 1.3299, -0.725712, -0.201635], 
        [0.0054086, 0.57299, 0.416415, 0.390883, -0.740232], 
        [2.06859, 1.87101, 1.98824, -2.91077, 0.0820638], 
        [0.902645, -0.257527, -0.0250149, 0.810974, -0.323619], 
        [0.492121, 1.02614, 1.27414, -0.775225, -0.154715]
      ], 
      [
        [0.284348, -1.02064, -1.4058, 2.01025, 2.43292], 
        [0.409564, -1.8065, -0.588854, -0.765913, -0.688296], 
        [0.538462, -0.64016, -1.24947, -0.245956, -0.869769], 
        [-0.054774, -0.820775, 0.462078, -0.0654275, -0.934235], 
        [-0.402937, 0.248906, -2.29374, 0.13392, 1.43473]
      ], 
      [
        [0.350459, 1.18417, -0.625226, -0.311076, 0.045635], 
        [0.323669, -0.676468, 1.43463, -0.305277, 0.183317], 
        [0.596093, 1.82243, 0.871849, -1.3162, -0.307149], 
        [-0.0467978, 0.521817, -0.311204, -1.09839, 0.989329], 
        [-0.32542, -0.634232, -1.63115, -0.963376, 1.11885]
      ], 
      [
        [-0.0626811, -0.343493, 1.61674, 2.00101, -1.99303], 
        [-1.28546, 1.36179, -0.663408, 0.432393, -0.621403], 
        [-1.92732, 0.777747, 0.89018, -3.17746, -0.118413], 
        [0.276645, 1.06601, -0.204408, 0.5386, -0.61766], 
        [2.05421, -1.36615, 0.0655545, -2.22941, -1.16415]
      ], 
      [
        [-0.844461, 0.380356, -0.689637, -0.21932, -0.975595], 
        [-0.686231, 0.0299028, -0.320714, 0.0748739, 0.225947], 
        [0.933338, -1.74332, 0.62677, 1.71103, -0.191209], 
        [0.502827, -0.16201, -0.595089, -0.290764, -1.05585], 
        [1.93996, 0.472283, 1.11893, 0.62859, -0.388826]
      ]
    ], 
    [
      [
        [-0.107905, 2.06343, -1.08616, -0.915616, 0.896901], 
        [0.253781, -0.364462, -0.346612, -0.627686, 0.0999118], 
        [0.305031, 0.836664, -0.0557397, -1.40849, -1.19675], 
        [-0.884139, 0.0599202, 0.595585, 0.954702, 0.564389], 
        [-0.985863, 0.850148, 0.14577, -0.49859, 0.867738]
      ], 
      [
        [-0.963193, 0.555834, -0.504522, 0.0620073, -2.31655], 
        [-0.142859, 2.68959, -1.89407, -0.249924, -0.784344], 
        [0.810684, 0.402282, 0.269791, 1.15916, -1.14448], 
        [0.0725691, -0.503327, 0.325401, 1.66103, -0.527198], 
        [-0.576783, 0.476453, 0.596458, 1.11169, 0.955192]
      ], 
      [
        [-0.0156668, -2.11, -0.0320946, -0.27069, 1.01751], 
        [-0.60644, 1.10474, -0.321379, -0.596434, -0.6732], 
        [0.365498, -0.108641, -1.5077, 0.821122, -0.244604], 
        [0.54329, 1.90383, -0.0412237, 1.26836, 0.982989], 
        [0.695276, -0.00972272, 0.126824, 0.537547, -0.68441]
      ], 
      [
        [1.24861, -0.477794, -1.52261, -1.18881, -0.801027], 
        [-0.0818397, -0.188448, -0.655533, -0.778766, 0.257865], 
        [-0.255535, 0.535765, 1.17231, 0.618329, 0.145949], 
        [0.851952, 0.432206, 0.0965555, 0.268556, -0.600678], 
        [1.42408, 0.144892, 0.744119, -0.255394, 0.825646]
      ], 
      [
        [-0.112375, 1.55427, -0.342305, -0.912158, -1.19006], 
        [-0.293987, 0.0247494, 0.280611, 0.723531, 1.85076], 
        [2.12018, 0.824188, 0.596618, -0.680421, -0.35468], 
        [0.749372, -0.296006, 0.0774152, -0.965204, 0.867055], 
        [0.122378, -0.476106, 0.587524, 0.491362, 1.97302]
      ]
    ]
  ]
]


# --- 2. Define Weight (Kernel) Tensor ---
# The weight tensor for Conv1d should have the shape: (C_out, C_in, K)
# C_out: Number of output channels (number of filters)
# C_in: Number of input channels (must match the input tensor)
# K: Kernel size (the width of the sliding window)

# Create a sample weight tensor (also known as the convolution kernel)
weight_tensor = [
  [
    [
      [
        [-0.516964, 1.22192, 0.721333], 
        [0.869636, 1.61822, 1.58856], 
        [-1.18831, -0.187125, -0.687744]
      ], 
      [
        [-0.0634201, 0.849739, -0.385892], 
        [0.297398, -1.05039, 1.32076], 
        [0.957779, -1.17469, 0.435903]
      ], 
      [
        [-1.55086, 0.884557, 0.0114265], 
        [-1.02266, -0.0083384, 0.233371], 
        [-0.849648, -0.524936, -0.663464]
      ]
    ], 
    [
      [
        [-0.566608, 1.01952, 0.122441], 
        [-0.335384, 0.824665, 2.24651], 
        [-0.391692, 0.0788633, 0.843509]
      ], 
      [
        [-0.899953, -0.31621, -1.06146], 
        [-1.66251, -0.215853, 1.08075], 
        [1.21028, -0.243826, 1.36622]
      ], 
      [
        [0.713993, -0.492485, -1.11412], 
        [-0.953354, -1.37982, 0.650501], 
        [-1.44723, -0.949764, 1.60879]
      ]
    ]
  ], 
  [
    [
      [
        [-0.669162, -1.6002, 0.020242], 
        [0.852093, 0.670735, -0.931605], 
        [0.398951, 0.113936, 0.563424]
      ], 
      [
        [1.95007, -1.64294, -0.600347], 
        [-0.112102, 0.837057, -2.69334], 
        [-1.89441, -1.05265, -0.420943]
      ], 
      [
        [0.57271, -1.2647, 0.837908], 
        [-0.430761, 0.212775, 0.603403], 
        [-0.127745, 1.35226, 0.430532]
      ]
    ], 
    [
      [
        [1.16225, 0.824048, -0.70246], 
        [1.04592, 1.42899, -0.167399], 
        [0.278383, 0.649517, -0.742612]
      ], 
      [
        [0.267558, 0.1339, -1.16002], 
        [-2.0782, -0.254266, -1.52884], 
        [0.0336459, -1.34662, -0.595901]
      ], 
      [
        [-1.69836, -0.242858, 0.575486], 
        [0.557238, 0.787936, -1.46632], 
        [-1.263, 1.64635, -1.18114]
      ]
    ]
  ], 
  [
    [
      [
        [-0.748972, -1.69712, 0.756976], 
        [-0.989778, -1.31725, 0.69877], 
        [-1.68927, 0.096982, -0.324205]
      ], 
      [
        [-0.0683886, -0.747694, -0.271226], 
        [-0.780501, 1.02708, -0.307149], 
        [-0.0467978, 0.521817, -0.311204]
      ], 
      [
        [-1.09839, 0.989329, -0.32542], 
        [-0.634232, -1.63115, -0.963376], 
        [1.11885, -0.0626811, -0.343493]
      ]
    ], 
    [
      [
        [1.61674, 2.00101, -1.99303], 
        [-1.28546, 1.36179, -0.663408], 
        [0.432393, -1.89407, -0.249924]
      ], 
      [
        [-0.784344, 0.810684, 0.402282], 
        [0.269791, 1.15916, -1.14448], 
        [0.0725691, -0.503327, 0.325401]
      ], 
      [
        [1.66103, -0.527198, -0.576783], 
        [0.476453, 0.596458, 1.11169], 
        [0.955192, -0.0156668, -2.11]
      ]
    ]
  ]
]

input_tensor = torch.tensor(input_tensor)
weight_tensor = torch.tensor(weight_tensor).permute(1, 0, 2, 3, 4)  # Fix: swap in/out channels

bias = torch.tensor([-0.516964, -1.02266, 1.21028])


# --- 3. Perform 1D Convolution ---
# We will use `torch.nn.functional.conv1d` for a direct functional call.
# This function is useful when you want to perform a convolution without defining a layer.
# It takes the input and weights as direct arguments.
# You can also specify other parameters like bias, stride, padding, etc.

# Perform the convolution operation
output_tensor = F.conv_transpose3d(
    input=input_tensor,
    weight=weight_tensor,
    bias=bias,
    stride=[2,2,2]
)

print("--- Output ---")
print(f"Shape: {output_tensor.shape}")
print(f"Tensor:\n{output_tensor}\n")

# --- Explanation of Output Shape ---
# The output length (L_out) is calculated as:
# L_out = floor((L_in + 2*padding - dilation*(kernel_size - 1) - 1) / stride + 1)
# In our case:
# L_out = floor((10 + 2*0 - 1*(3 - 1) - 1) / 1 + 1)
# L_out = floor((10 - 2 - 1) / 1 + 1)
# L_out = floor(7 + 1) = 8
#
# So the expected output shape is (batch_size, output_channels, 8), which is (1, 2, 8).


--- Output ---
Shape: torch.Size([3, 3, 11, 11, 11])
Tensor:
tensor([[[[[-4.9629e-01, -7.0497e-01, -2.0582e+00,  ..., -6.3485e-01,
             1.3805e+00,  6.4072e-01],
           [-1.1125e+00, -9.9465e-01,  3.5291e-01,  ...,  3.1602e+00,
             2.0371e+00,  1.8777e+00],
           [-7.2953e-01,  1.2586e+00, -4.6696e-01,  ..., -2.2363e+00,
            -2.5262e+00, -1.9366e+00],
           ...,
           [-1.2017e+00, -1.5430e+00, -6.7448e-01,  ...,  6.9755e-01,
            -1.7049e+00, -3.3135e+00],
           [-2.2246e-01, -1.1982e+00, -3.1825e+00,  ..., -3.3891e+00,
            -2.4316e+00, -5.2012e-01],
           [-1.9820e-01, -5.8602e-01, -1.6471e+00,  ...,  1.3480e+00,
            -6.8061e-02,  1.8559e+00]],

          [[-8.7582e-01, -1.0939e+00, -1.7938e+00,  ..., -1.2920e+00,
             8.8287e-01, -1.0583e+00],
           [-1.3942e+00, -6.7886e-02, -2.0968e+00,  ...,  1.6579e+00,
            -2.1998e+00,  1.5357e+00],
           [-3.2439e-01,  1.4260e+00,  1.2001e+00

In [21]:
(1.48304 +  0.977345 +  3.21028 +  2.56342)/4
import math
math.sqrt(0.6008240183621112)

0.7751283883087441

In [20]:
(1.48304 ** 2+ 3.22192 ** 2+ 2.72133 ** 2+ 2.86964  ** 2+ 0.977345 ** 2+ 1.99166 ** 2+ 2.23337 ** 2+ 1.15035 ** 2)/8 - 2.0810818749999997**2 

0.6008240183621112

In [34]:
a = [
  [
    [0.37454, 0.796543, 0.950714, 0.183435],
    [0.115055, 0.496861, 0.609066, 0.102915],
    [0.834842, 0.432542, 0.104796, 0.0317748],
    [0.989011, 0.304536, 0.549545, 0.67148]
  ],
  [
    [0.783832, 0.258047, 0.634834, 0.684217],
    [0.113488, 0.851757, 0.974483, 0.909671],
    [0.0174903, 0.426484, 0.891573, 0.266471],
    [0.300964, 0.473737, 0.247062, 0.761432]
  ]
]
b = [
  [
    [-1.03393, 2.44384, 1.44267, 1.73927],
    [-2.04531, -0.0166768, 0.466742, -1.6993],
    [2.42056, -0.487653, 2.73244, 1.42799], 
    [1.12685, 3.90014, -3.28587, -1.20069]
  ],
  [
    [1.6481, -1.40492, 2.09184, 2.85799], 
    [1.11448, 1.57587, -2.93264, -2.526],
    [-0.614297, -0.0935957, 1.04363, -0.622409],
    [-3.78815, -0.499848, -1.56869, 1.62137]
  ]
]

a = torch.tensor(a,requires_grad=True)
b = torch.tensor(b, requires_grad=True)
 
out = a.exp()

out.sum().backward(retain_graph=True)
out.sum().backward()

a.grad, b.grad


(tensor([[[2.9086, 4.4357, 5.1751, 2.4027],
          [2.2439, 3.2871, 3.6774, 2.2168],
          [4.6089, 3.0823, 2.2210, 2.0646],
          [5.3771, 2.7120, 3.4649, 3.9143]],
 
         [[4.3797, 2.5888, 3.7734, 3.9644],
          [2.2404, 4.6875, 5.2996, 4.9670],
          [2.0353, 3.0637, 4.8779, 2.6107],
          [2.7023, 3.2120, 2.5605, 4.2827]]]),
 None)